# Predicting an Protein–ATP Complex with AlphaFold 3 in Google Colab

Based on [ColabFold-Boltz](https://github.com/sokrypton/ColabFold)

This notebook demonstrates how to use **Google Colab** to to predict the 3‑D structure of protein bound to the ligand **ATP** using ColabFold, a open-sourced user-friendly scripts which incorporated all curently commonly seen prediction tools, such as **[AlphaFold 2 (AF2)](https://github.com/google-deepmind/alphafold)**, [Boltz](https://github.com/jwohlwend/boltz) (a MIT-recreated open sourced AF3). Based on the predicted structure, we can come up with hypothesis of if this binding could be real or not. It covers:

1. Intro to Colab, Jupyter, MSAs & AlphaFold-type of predictors.
2. Preparing an input FASTA that specifies both protein & ligand
3. Generating a Multiple Sequence Alignment (MSA) with MMseqs2
4. Running AF3 inference (with pretrained weights)
5. Visualising the resulting complex in‑notebook


# This is a work-in-progress notebook for [Boltz](https://github.com/jwohlwend/boltz)

⚠️ **Warning to Users:**
- **Alpha Version:** This notebook is currently under active development and is considered a beta version.
- **Usage at Your Own Risk:** Use this notebook at your own discretion and risk.

## 1. What are Google Colab & Jupyter notebooks?
- **Jupyter notebook**: an interactive document that combines live code, formatted text, math, and plots. Each *cell (or code block)* can be executed individually.
- **Google Colab**:
  * Google’s free cloud hosting of Jupyter notebooks.
  * No local setup (aka you do not need to borther with learning how to install any software);
  * free CPUs/GPUs (by copying this colab to your own drive, you request a cpu/gpu memory resource from google);
  * notebooks auto‑save to Google Drive.

📌 **Usage tips**  
- Copy to your own google drive using **File -> Save as a copy in drive**
- Run a cell with **Shift + Enter** or click the ▶ icon.  
- Enable a GPU under **Runtime ▸ Change runtime type**, use T4/GPU. AF3-type of predictors needs to be ran on GPU, so change to a GPU.
- Your runtime is temporary—save important files to Drive before disconnects.
- use the down arraw to see the source code

## 2 Core concepts
### 2.1 Multiple Sequence Alignment (MSA)
- [what is a MSA?](https://www.youtube.com/watch?v=TZaA_-4j19w)
- Aligns homologous sequences column‑wise; reveals conserved, functionally important residues used by AlphaFold.

### 2.2 AlphaFold 3 (AF3)
- [A 5 min peek of AlphaFold](https://www.youtube.com/watch?v=7q8Uw3rmXyE)
- [A comprehensive 1-hour discussion of AlphaFold3 by Sergey O](https://www.youtube.com/watch?v=qjFgthkKxcA)
- Use Next‑gen model (here use AlphaFold3) that predicts full complexes: proteins + DNA/RNA + **small molecule ligands**.

### 2.3 ATP & ATPases
- ATP (adenosine triphosphate) is the cell’s energy currency. **ATPases** are enzymes that hydrolyse ATP, using the released energy for work.
- Designing ATP-binding or ATP-hydrolyzing protein will allow we generate fuels for protein nanomachine (another 2016 Nobel prize-winning hot field).

In [ ]:
#@title Install dependencies
%%time
import os
if not os.path.isfile("COLABFOLD_READY"):
  print("installing colabfold...")
  os.system("pip install -q --no-warn-conflicts 'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
  if os.environ.get('TPU_NAME', False) != False:
    os.system("pip uninstall -y jax jaxlib")
    os.system("pip install --no-warn-conflicts --upgrade dm-haiku==0.0.10 'jax[cuda12_pip]'==0.3.25 -f https://storage.googleapis.com/jax-releases/jax_cuda_releases.html")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
  os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
  os.system("touch COLABFOLD_READY")

if not os.path.isfile("BOLZ_READY"):
  os.system("apt-get install -y aria2")
  os.system("pip install -q --no-warn-conflicts boltz")
  os.system("mkdir weights")
  os.system("aria2c -d weights -x8 -s8 https://colabfold.steineggerlab.workers.dev/boltz1.ckpt")
  os.system("aria2c -d weights -x8 -s8 https://colabfold.steineggerlab.workers.dev/ccd.pkl")
  os.system("touch BOLZ_READY")

installing colabfold...
CPU times: user 910 ms, sys: 170 ms, total: 1.08 s
Wall time: 5min 29s


In [ ]:
#@title Reinstall torch to remove a bug from public version (LA)
# restart will be needed after reinstallation
# REMOVE every old build first
!pip uninstall -y torch torchvision torchaudio

# ---- GPU, CUDA 11.8 example ----
!pip install --pre torch torchvision torchaudio \
     --index-url https://download.pytorch.org/whl/nightly/cu118
# ---- GPU, CUDA 12.1 example ----
# pip install --pre torch torchvision torchaudio \
#      --index-url https://download.pytorch.org/whl/nightly/cu121
# ---- CPU fallback ----
# pip install --pre torch torchvision torchaudio \
#      --index-url https://download.pytorch.org/whl/nightly/cpu

In [ ]:
#@title Input protein sequence(s), then hit `Runtime` -> `Run all`
from google.colab import files
import os
import re
import hashlib
import random
import requests
from string import ascii_uppercase

# Function to add a hash to the jobname
def add_hash(x, y):
    return x + "_" + hashlib.sha1(y.encode()).hexdigest()[:5]

# User inputs
query_sequence = 'SPRLEENTDRIIEAIRAGKIPRSVLEEWRRRSREDSDAMTPEERRALEEAHEAGEVEITGETFQVPEVPWLYYGRGVYEKFGRRLTGTLSLTGEPIDTPLRESREVSRRIDAALDEVEA'  #@param {type:"string"}
#@markdown  - Use `:` to specify inter-protein chainbreaks for **modeling complexes** (supports homo- and hetro-oligomers). For example **PI...SK:PI...SK** for a homodimer
ligand_input = 'C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N'  #@param {type:"string"}
#@markdown  - Use `:` to specify multiple ligands as smile strings
ligand_input_ccd = 'ATP'  #@param {type:"string"}
#@markdown - Use `:` to specify multiple ligands as CCD codes (three-letter codes)
ligand_input_common_name = ''  #@param {type:"string"}
#@markdown - Use `:` to specify multiple ligands with their common name (e.g. Aspirin; SMILES fetched from [PubChem](https://pubchem.ncbi.nlm.nih.gov) API)
dna_input = ''  #@param {type:"string"}
#@markdown - Use `:` to specify multiple DNA sequences
jobname = 'test'  #@param {type:"string"}

# Clean up the query sequence and jobname
query_sequence = "".join(query_sequence.split())
ligand_input = "".join(ligand_input.split())
ligand_input_ccd = "".join(ligand_input_ccd.split())
ligand_input_common_name = "".join(ligand_input_common_name.split())
dna_input = "".join(dna_input.split())
basejobname = "".join(jobname.split())
basejobname = re.sub(r'\W+', '', basejobname)
jobname = add_hash(basejobname, query_sequence)

# Check if a directory with jobname exists
def check(folder):
    return not os.path.exists(folder)

if not check(jobname):
    n = 0
    while not check(f"{jobname}_{n}"):
        n += 1
    jobname = f"{jobname}_{n}"

# Make directory to save results
os.makedirs(jobname, exist_ok=True)

from string import ascii_uppercase

# Split sequences on chain breaks
protein_sequences = query_sequence.strip().split(':') if query_sequence.strip() else []
ligand_sequences = ligand_input.strip().split(':') if ligand_input.strip() else []
ligand_sequences_ccd = ligand_input_ccd.strip().split(':') if ligand_input_ccd.strip() else []
ligand_sequences_common_name = ligand_input_common_name.strip().split(':') if ligand_input_common_name.strip() else []
dna_sequences = dna_input.strip().split(':') if dna_input.strip() else []

def get_smiles(compound_name):
    autocomplete_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/autocomplete/compound/{compound_name}/json?limit=1"
    autocomplete_response = requests.get(autocomplete_url)
    if autocomplete_response.status_code != 200:
        return None

    autocomplete_data = autocomplete_response.json()
    if autocomplete_data.get("status", {}).get("code") != 0 or autocomplete_data.get("total", 0) == 0:
        return None

    suggested_compound = autocomplete_data.get("dictionary_terms", {}).get("compound", [])
    if not suggested_compound:
        return None
    suggested_compound_name = suggested_compound[0]

    smiles_url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{suggested_compound_name}/property/CanonicalSMILES/JSON"
    smiles_response = requests.get(smiles_url)
    if smiles_response.status_code != 200:
        return None

    smiles_data = smiles_response.json()
    properties = smiles_data.get("PropertyTable", {}).get("Properties", [])
    if len(properties) == 0:
        return None

    return properties[0].get("CanonicalSMILES")

smiles_cache = {}
for name in ligand_sequences_common_name:
    if name not in smiles_cache:
        smiles_cache[name] = get_smiles(name)
        if smiles_cache[name] is not None:
          print(f"Mapped compound {name} to {smiles_cache[name]}")

    if smiles_cache[name] is not None:
        ligand_sequences.append(smiles_cache[name])

# Initialize chain labels starting from 'A'
chain_labels = iter(ascii_uppercase)

fasta_entries = []
csv_entries = []
chain_label_to_seq_id = {}
seq_to_seq_id = {}
seq_id_counter = 0  # Counter for unique sequences

# Process protein sequences
for seq in protein_sequences:
    seq = seq.strip()
    if not seq:
        continue  # Skip empty sequences
    chain_label = next(chain_labels)
    # Check if sequence has been seen before
    if seq in seq_to_seq_id:
        seq_id = seq_to_seq_id[seq]
    else:
        seq_id = f"{jobname}_{seq_id_counter}"
        seq_to_seq_id[seq] = seq_id
        seq_id_counter += 1
        # For CSV file (for ColabFold), add only unique sequences
        csv_entries.append((seq_id, seq))
    chain_label_to_seq_id[chain_label] = seq_id
    # For FASTA file
    msa_path = os.path.join(jobname, f"{seq_id}.a3m")
    header = f">{chain_label}|protein|{msa_path}"
    sequence = seq
    fasta_entries.append((header, sequence))

# Process ligand sequences (assumed to be SMILES strings)
for lig in ligand_sequences:
    lig = lig.strip()
    if not lig:
        continue  # Skip empty ligands
    chain_label = next(chain_labels)
    lig_type = 'smiles'
    header = f">{chain_label}|{lig_type}"
    sequence = lig
    fasta_entries.append((header, sequence))

# Process DNA sequences (NO MSA is generated)
for seq in dna_sequences:
    seq = seq.strip()
    if not seq:
        continue  # Skip empty sequences
    chain_label = next(chain_labels)
    lig_type = 'DNA'
    header = f">{chain_label}|{lig_type}"
    sequence = seq
    fasta_entries.append((header, sequence))

# Process ligand sequences (CCD codes)
for lig in ligand_sequences_ccd:
    lig = lig.strip()
    if not lig:
        continue  # Skip empty ligands
    chain_label = next(chain_labels)
    lig_type = 'ccd'
    header = f">{chain_label}|{lig_type}"
    sequence = lig.upper()  # Ensure CCD codes are uppercase
    fasta_entries.append((header, sequence))

# Write the CSV file for ColabFold
queries_path = os.path.join(jobname, f"{jobname}.csv")
with open(queries_path, "w") as text_file:
    text_file.write("id,sequence\n")
    for seq_id, seq in csv_entries:
        text_file.write(f"{seq_id},{seq}\n")

# Write the FASTA file
queries_fasta = os.path.join(jobname, f"{jobname}.fasta")
with open(queries_fasta, 'w') as f:
    for header, sequence in fasta_entries:
        f.write(f"{header}\n{sequence}\n")

# Optionally, print the output for verification
#print(f"Generated FASTA file '{queries_fasta}':\n")
#for header, sequence in fasta_entries:
#    print(f"{header}\n{sequence}\n")


In [ ]:
# check the installations
import torch, torchvision, platform, sys, subprocess, os
print("torch :", torch.__version__) # should be higher than 2.4.x
print("torchvision:", torchvision.__version__)
print("CUDA in torch:", torch.version.cuda)
print("torch is at :", torch.__file__)
print("torchvision is at :", torchvision.__file__)
import torch.library              # this should have custom_op now
print(hasattr(torch.library,"custom_op"))  # should give True



torch : 2.8.0.dev20250609+cu118
torchvision: 0.23.0.dev20250609+cu118
CUDA in torch: 11.8
torch is at : /usr/local/lib/python3.11/dist-packages/torch/__init__.py
torchvision is at : /usr/local/lib/python3.11/dist-packages/torchvision/__init__.py
True


In [ ]:
#@title Generate MSA with ColabFold
!colabfold_batch "{queries_path}" "{jobname}" --msa-only

2025-06-19 22:09:32,380 Running colabfold 1.5.5 (64c8b2fe2ecf3199bc2d3c60b5da4b929a41086e)

limited shared resource only capable of processing a few thousand MSAs per day. Please
submit jobs only from a single IP address. We reserve the right to limit access to the
server case-by-case when usage exceeds fair use. If you require more MSAs: You can 
precompute all MSAs with `colabfold_search` or host your own API and pass it to `--host-url`

2025-06-19 22:09:40,867 Running on GPU
2025-06-19 22:09:41,117 Found 4 citations for tools or databases
2025-06-19 22:09:41,117 Query 1/1: test_b1c13_3_0 (length 119)
COMPLETE: 100% 150/150 [00:04<00:00, 36.61it/s]
2025-06-19 22:09:45,217 Saved test_b1c13_3/test_b1c13_3_0.pickle
2025-06-19 22:09:45,878 Done


In [ ]:
!ls # see what are the files generated, you should be able to see your job directories

alphafold   colabfold	     sample_data  test_b1c13_0	test_b1c13_2  weights
BOLZ_READY  COLABFOLD_READY  test_b1c13   test_b1c13_1	test_b1c13_3


In [ ]:
queries_path # this should be the msa

'test_b1c13_3/test_b1c13_3.csv'

In [ ]:
!cat $queries_path # check the msa content, if you used the provided sequence, then you should only see 2 lines. Think about it, does it fit the expected MSA format you read above?

id,sequence
test_b1c13_3_0,SPRLEENTDRIIEAIRAGKIPRSVLEEWRRRSREDSDAMTPEERRALEEAHEAGEVEITGETFQVPEVPWLYYGRGVYEKFGRRLTGTLSLTGEPIDTPLRESREVSRRIDAALDEVEA


In [ ]:
jobname

'test_b1c13_2'

In [ ]:
jobname # check the running job

'test_b1c13_3'

In [ ]:
!ls # check the files in the folder

alphafold   colabfold	     sample_data  test_b1c13_0	test_b1c13_2  weights
BOLZ_READY  COLABFOLD_READY  test_b1c13   test_b1c13_1	test_b1c13_3


In [ ]:
#@title Predict structure using boltz
!boltz predict --out_dir "{jobname}" "{jobname}/{jobname}.fasta" --cache weights`

Checking input data.
Found 0 existing processed inputs, skipping them.
Processing 1 inputs with 1 threads.
100% 1/1 [00:00<00:00,  6.72it/s]
Using bfloat16 Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
Running structure prediction for 1 input.
/usr/local/lib/python3.11/dist-packages/pytorch_lightning/utilities/migration/utils.py:56: The loaded checkpoint was produced with Lightning v2.5.0.post0, which is newer than your current Lightning version: v2.5.0
2025-06-19 22:11:54.989303: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Predicting DataLoader 0: 100% 1/1 [00:24<00:00, 24.89s/it]Number of failed examples: 0

In [ ]:
!ls $jobname/boltz_results_$jobname/predictions/$jobname/ # check the content of the job folder, the `cif` file is the prediction results.

confidence_test_b1c13_3_model_0.json  plddt_test_b1c13_3_model_0.npz
pae_test_b1c13_3_model_0.npz	      test_b1c13_3_model_0.cif
pde_test_b1c13_3_model_0.npz


In [ ]:
#@title Visialize the results using Pymol
# install pymol
!apt-get update -y        # -qq keeps the log short
!apt-get install -y pymol # open-source PyMOL 2.x
!pip -q install py3Dmol
!pip install icecream # this is a package commonly used for debugging

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:3 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,798 kB]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,051 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,747 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.

In [ ]:
import io
import py3Dmol
from IPython.display import display

pred_results = os.path.join(jobname, f"boltz_results_{jobname}", "predictions", jobname, f"{jobname}_model_0.cif")
assert os.path.isfile(pred_results) # verify the result exists

with open(pred_results,'r') as f:
  pdb_text = f.read()
print(len(pdb_text))
# Push that PDB string into py3Dmol and show a cartoon
view = py3Dmol.view(width=500, height=450)
view.addModel(pdb_text, 'cif')      # just pass the string
view.setStyle({'cartoon': {}})
view.addStyle({'chain': 'B', 'hetflag': True},  # selection
              {'stick': {'radius': 0.25, 'colorscheme': 'element'}})
view.zoomTo()
view.show()

90224


3Dmol.js failed to load for some reason. Please check your browser console for error messages.

- Questions:
1. Does the procedure look reasonable to you? If something feels funky, what is it?
2. Why does the MSA only have 1 line?
3. Does the results looking like something really interacting with ATP?
4. Feel free to change the sequences to others to try again.